# Small Llama — Sentence Completion Sanity Check

Qualitative check requested by Kurfali: beyond el-BLiMP scoring, can Small
Llama produce fluent, plausible Greek text when generating freely?

Runs the 100% checkpoint (best-trained) on a handful of sentence starts,
generating multiple completions per prompt to see typical output quality.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"GPU available: {torch.cuda.is_available()}")

GPU available: False


## Load the 100% checkpoint\n\nAdjust the path if your checkpoint lives somewhere else.

In [4]:
MODEL_DIR = "/content/drive/MyDrive/Thesis/models/small_llama_batches/small_llama_100pct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=torch.float16)
if torch.cuda.is_available():
    model = model.to('cuda')
model.eval()
print("Model loaded")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/75 [00:00<?, ?it/s]

Model loaded


## Generation function

In [5]:
def generate_completion(model, tokenizer, prompt, max_new_tokens=25, temperature=0.8, top_p=0.9):
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

## Run completions\n\nA handful of simple, common sentence-start prompts, 3 completions each (since sampling means outputs vary run to run) -- pick the most representative/coherent one(s) for your thesis text.

In [6]:
PROMPTS = [
    "Η Ελλάδα είναι μια χώρα",        # "Greece is a country..."
    "Χθες πήγα στο",                    # "Yesterday I went to the..."
    "Ο καιρός σήμερα είναι",            # "The weather today is..."
    "Το σπίτι μου βρίσκεται",           # "My house is located..."
    "Οι άνθρωποι στην Αθήνα",           # "The people in Athens..."
]

N_COMPLETIONS_PER_PROMPT = 3

results = []
for prompt in PROMPTS:
    print(f"\n{'='*70}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*70}")
    for i in range(N_COMPLETIONS_PER_PROMPT):
        completion = generate_completion(model, tokenizer, prompt)
        print(f"  [{i+1}] {completion}")
        results.append({'prompt': prompt, 'completion': completion})


PROMPT: Η Ελλάδα είναι μια χώρα
  [1] Η Ελλάδα είναι μια χώρα, που έχει έδρα το Γκαλάτσι, την πρωτεύουσα της επαρχίας της Φλώρινας, που ανήκει στη
  [2] Η Ελλάδα είναι μια χώρα (ή κωμόπολη στα ελληνικά) είναι το μεγαλύτερο από τα τρία μεγαλύτερα προάστια της Ελλάδας
  [3] Η Ελλάδα είναι μια χώρα, η οποία είναι μέλος της Ευρωπαϊκής Ένωσης. Το όνομα είναι το ομώνυμο νησί της Ελλάδας και χρησιμοποιείται από το Ευρωπαϊκό Δικ

PROMPT: Χθες πήγα στο
  [1] Χθες πήγα στοtυγκράτης (HWV 7) ήταν ο τίτλος του γερμανικού διπλού δίσκου του Α
  [2] Χθες πήγα στο (στη γλώσσα των λέξεων Χθες) είναι ο τίτλος μιας κινηματογραφικής ταινίας του
  [3] Χθες πήγα στο είναι ο τίτλος της συλλογής τραγουδιών του γερμανικού metal συγκροτήματος Kreator. Κυκλοφόρησε το

PROMPT: Ο καιρός σήμερα είναι
  [1] Ο καιρός σήμερα είναι κατάλογος με τις καλύτερες επιδόσεις στον παγκόσμιο ραδιοτηλεοπτικό οργανισμό. Η εκπομπή αυτή είναι η μεγαλύτερη
  [2] Ο καιρός σήμερα είναι κατάλογος με τα σπορ του έθνους. Κατάλογος Παραπο

## Save results for reference

In [7]:
import json
from pathlib import Path

OUTPUT_PATH = Path("/content/drive/MyDrive/Thesis/results/small_llama_sentence_completions.json")
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Saved {len(results)} completions to {OUTPUT_PATH}")
print("\nReview the outputs above and pick 2-3 representative examples")
print("(one clearly coherent, and one or two showing typical/weaker output)")
print("to include as qualitative examples in Section 3.4.4 of the thesis.")

Saved 15 completions to /content/drive/MyDrive/Thesis/results/small_llama_sentence_completions.json

Review the outputs above and pick 2-3 representative examples
(one clearly coherent, and one or two showing typical/weaker output)
to include as qualitative examples in Section 3.4.4 of the thesis.
